# Final Optuna LGBM

Run the final `Optuna` tuning after the data-side experiments are fixed. The notebook uses the best sparsity threshold from `08_sparsity_thresholds_lgbm.ipynb` artifacts when available, otherwise it falls back to `0.985`.

In [1]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd

/Users/ayeustsihneyeu/ml_/santander/.santander/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, train_test_split

from src.features import add_rowwise_features, get_sparse_columns_to_keep
from src.loader import Loader
from src.metrics import format_metric_value
from src.modeling import build_lgbm_regressor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
N_TRIALS = 20
FALLBACK_SPARSITY_THRESHOLD = 0.985

ARTIFACTS_DIR = Path("../artifacts/final_optuna_lgbm")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## Load Data

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

## Fixed Data Setup

In [6]:
def load_best_sparsity_threshold() -> float | None:
    results_path = Path("../artifacts/sparsity_thresholds_rowwise_lgbm/sparsity_threshold_results.csv")
    if not results_path.exists():
        return FALLBACK_SPARSITY_THRESHOLD

    results = pd.read_csv(results_path).sort_values("cv_rmsle_mean")
    threshold = results.iloc[0]["threshold"]
    if str(threshold).lower() == "none":
        return None
    return float(threshold)


SPARSITY_THRESHOLD = load_best_sparsity_threshold()
SPARSITY_THRESHOLD

0.9875

In [8]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

## Optuna Search

The objective optimizes RMSE in log-space, which is equivalent to `RMSLE` for the `log1p(target)` setup. Sparse filtering is fitted inside each fold only on that fold's training data.

In [9]:
def suggest_params(trial: optuna.Trial) -> dict:
    return {
        "n_estimators": trial.suggest_int("n_estimators", 250, 1600),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.04, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 90),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.45, 0.95),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5),
    }

In [10]:
def objective(trial: optuna.Trial) -> float:
    params = suggest_params(trial)
    fold_scores = []

    for fold_train_idx, fold_valid_idx in cv.split(X_train_raw, y_train_log):
        X_fold_train_raw = X_train_raw.iloc[fold_train_idx]
        X_fold_valid_raw = X_train_raw.iloc[fold_valid_idx]
        y_fold_train_log = y_train_log.iloc[fold_train_idx]
        y_fold_valid_log = y_train_log.iloc[fold_valid_idx]

        keep_columns = get_sparse_columns_to_keep(X_fold_train_raw, SPARSITY_THRESHOLD)
        X_fold_train = add_rowwise_features(X_fold_train_raw[keep_columns])
        X_fold_valid = add_rowwise_features(X_fold_valid_raw[keep_columns])

        model = build_lgbm_regressor(params)
        model.fit(X_fold_train, y_fold_train_log)
        y_fold_pred_log = model.predict(X_fold_valid)
        fold_scores.append(root_mean_squared_error(y_fold_valid_log, y_fold_pred_log))

    return float(np.mean(fold_scores))

In [11]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

study.best_value

[I 2026-05-04 10:10:39,060] A new study created in memory with name: no-name-7ecc895c-faad-4ebc-98bc-d2b0b64f8ba5
Best trial: 0. Best value: 1.40947:   5%|▌         | 1/20 [00:30<09:37, 30.41s/it]

[I 2026-05-04 10:11:09,486] Trial 0 finished with value: 1.4094691248406044 and parameters: {'n_estimators': 756, 'learning_rate': 0.036103605491130335, 'num_leaves': 98, 'max_depth': 9, 'min_child_samples': 18, 'subsample': 0.662397808134481, 'subsample_freq': 1, 'colsample_bytree': 0.8830880728874676, 'reg_alpha': 0.10129197956845731, 'reg_lambda': 0.34702669886504117, 'min_split_gain': 0.010292247147901223}. Best is trial 0 with value: 1.4094691248406044.


Best trial: 0. Best value: 1.40947:  10%|█         | 2/20 [01:08<10:31, 35.07s/it]

[I 2026-05-04 10:11:47,811] Trial 1 finished with value: 1.4137646354889593 and parameters: {'n_estimators': 1560, 'learning_rate': 0.02823193321466291, 'num_leaves': 39, 'max_depth': 6, 'min_child_samples': 20, 'subsample': 0.721696897183815, 'subsample_freq': 4, 'colsample_bytree': 0.6659725093210579, 'reg_alpha': 0.0028585493941961923, 'reg_lambda': 0.11462107403425029, 'min_split_gain': 0.06974693032602092}. Best is trial 0 with value: 1.4094691248406044.


Best trial: 2. Best value: 1.36537:  15%|█▌        | 3/20 [01:49<10:38, 37.57s/it]

...

Best trial: 14. Best value: 1.35691:  95%|█████████▌| 19/20 [12:01<00:56, 56.17s/it]

[I 2026-05-04 10:22:40,189] Trial 18 finished with value: 1.3666322197056324 and parameters: {'n_estimators': 1283, 'learning_rate': 0.007706596726017606, 'num_leaves': 82, 'max_depth': 7, 'min_child_samples': 26, 'subsample': 0.9153175069788861, 'subsample_freq': 2, 'colsample_bytree': 0.7699212859905155, 'reg_alpha': 2.1399459095611584, 'reg_lambda': 0.004720765763422634, 'min_split_gain': 0.2336164113132571}. Best is trial 14 with value: 1.3569147480121722.


Best trial: 14. Best value: 1.35691: 100%|██████████| 20/20 [12:51<00:00, 38.56s/it]

[I 2026-05-04 10:23:30,281] Trial 19 finished with value: 1.3725063939277988 and parameters: {'n_estimators': 804, 'learning_rate': 0.013852861888916245, 'num_leaves': 96, 'max_depth': 10, 'min_child_samples': 11, 'subsample': 0.9095015737213298, 'subsample_freq': 3, 'colsample_bytree': 0.7602593397596824, 'reg_alpha': 2.4724927847844076, 'reg_lambda': 0.0004801865703329159, 'min_split_gain': 0.12616747474060067}. Best is trial 14 with value: 1.3569147480121722.


1.3569147480121722

In [12]:
best_params = study.best_params
best_params

{'n_estimators': 497,
 'learning_rate': 0.008202584804480604,
 'num_leaves': 81,
 'max_depth': 11,
 'min_child_samples': 5,
 'subsample': 0.8515393462995914,
 'subsample_freq': 3,
 'colsample_bytree': 0.766176819558098,
 'reg_alpha': 1.5556246271648941,
 'reg_lambda': 0.0011459234690630842,
 'min_split_gain': 0.22671541226113573}

## Holdout Check

In [ ]:
keep_columns = get_sparse_columns_to_keep(X_train_raw, SPARSITY_THRESHOLD)
X_train_final = add_rowwise_features(X_train_raw[keep_columns])
X_test_final = add_rowwise_features(X_test_raw[keep_columns])

final_model = build_lgbm_regressor(best_params)
final_model.fit(X_train_final, y_train_log)

y_pred_log = final_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [16]:
metrics = pd.DataFrame(
    {
        "metric": [
            "sparsity_threshold",
            "base_features_kept",
            "final_features",
            "cv_rmsle_mean",
            "holdout_rmsle",
            "holdout_rmse",
            "holdout_mae",
            "holdout_r2",
        ],
        "value": [
            "none" if SPARSITY_THRESHOLD is None else SPARSITY_THRESHOLD,
            len(keep_columns),
            X_train_final.shape[1],
            study.best_value,
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": format_metric_value}).hide(axis="index")

metric,value
sparsity_threshold,0.9875
base_features_kept,"2,482"
final_features,"2,490"
cv_rmsle_mean,1.3569
holdout_rmsle,1.3786
holdout_rmse,"6,948,547.5890"
holdout_mae,"3,904,504.4346"
holdout_r2,0.2435


## Save Artifacts

In [17]:
trials_df = study.trials_dataframe().sort_values("value")
trials_df.to_csv(ARTIFACTS_DIR / "final_optuna_trials.csv", index=False)
metrics.to_csv(ARTIFACTS_DIR / "final_optuna_metrics.csv", index=False)

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "optimizer": "optuna",
    "n_trials": N_TRIALS,
    "cv_folds": CV,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "sparsity_threshold": "none" if SPARSITY_THRESHOLD is None else SPARSITY_THRESHOLD,
    "rowwise_features": [
        "non_zero_count",
        "non_zero_ratio",
        "row_sum",
        "row_mean",
        "row_std",
        "row_max",
        "nz_mean",
        "nz_std",
    ],
    "base_features_kept": len(keep_columns),
    "final_features": X_train_final.shape[1],
    "best_cv_rmsle": float(study.best_value),
    "best_params": best_params,
    "holdout_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
    "holdout_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
    "holdout_mae": float(mean_absolute_error(y_test_raw, y_pred)),
    "holdout_r2": float(r2_score(y_test_raw, y_pred)),
}

with open(ARTIFACTS_DIR / "final_optuna_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(ARTIFACTS_DIR / "selected_base_features.json", "w") as f:
    json.dump(keep_columns, f, indent=2)

summary

{'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'optimizer': 'optuna',
 'n_trials': 20,
 'cv_folds': 5,
 'test_size': 0.33,
 'seed': 42,
 'sparsity_threshold': 0.9875,
 'rowwise_features': ['non_zero_count',
  'non_zero_ratio',
  'row_sum',
  'row_mean',
  'row_std',
  'row_max',
  'nz_mean',
  'nz_std'],
 'base_features_kept': 2482,
 'final_features': 2490,
 'best_cv_rmsle': 1.3569147480121722,
 'best_params': {'n_estimators': 497,
  'learning_rate': 0.008202584804480604,
  'num_leaves': 81,
  'max_depth': 11,
  'min_child_samples': 5,
  'subsample': 0.8515393462995914,
  'subsample_freq': 3,
  'colsample_bytree': 0.766176819558098,
  'reg_alpha': 1.5556246271648941,
  'reg_lambda': 0.0011459234690630842,
  'min_split_gain': 0.22671541226113573},
 'holdout_rmsle': 1.3786188512434583,
 'holdout_rmse': 6948547.589042924,
 'holdout_mae': 3904504.434609513,
 'holdout_r2': 0.24350865800602584}

## How To Read The Result

- Compare `best_cv_rmsle` against the best row-wise sparsity result from notebook `08`.
- Treat `holdout_rmsle` as a sanity check, not as the optimization target.
- If this wins, use this configuration as the tuned single-model candidate for final ensembling or repeated-CV validation.